# 02 · Exploratory Data Analysis (EDA)
**Question:** *Who misses appointments, when, and why?*

Sections
1. Overall no-show rate
2. Waiting time
3. Age & gender
4. Day of week & booking hour
5. Health conditions & welfare programme
6. SMS reminders (a surprising result)
7. Neighbourhoods
8. Patient history
9. Trend over time
10. Correlations
11. Statistical tests
12. Key findings

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

df = pd.read_csv("../data/processed/appointments_clean.csv", parse_dates=["scheduled_dt", "appointment_date", "scheduled_date"])
import sys; sys.path.insert(0, "../src"); import data_prep as dp
df["age_group"] = pd.Categorical(df["age_group"], categories=dp.AGE_LABELS, ordered=True)
df["wait_group"] = pd.Categorical(df["wait_group"], categories=dp.WAIT_LABELS, ordered=True)

BLUE, RED, GREY = "#2E86AB", "#E4572E", "#9AA5B1"
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold"})
FIG = "../reports/figures/"
BASE = df["missed"].mean()

def save(name):
    plt.tight_layout(); plt.savefig(FIG + name + ".png", dpi=160, bbox_inches="tight"); plt.show()

def rate_bar(col, title, order=None, rot=0, fname=None, figsize=(8, 4.2), annotate=True):
    g = df.groupby(col, observed=True)["missed"].agg(["mean", "size"])
    if order is not None: g = g.loc[order]
    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar(g.index.astype(str), g["mean"] * 100, color=BLUE)
    ax.axhline(BASE * 100, color=RED, ls="--", lw=1.5, label=f"Overall {BASE:.1%}")
    if annotate:
        for b, (m, n) in zip(bars, zip(g["mean"], g["size"])):
            ax.text(b.get_x() + b.get_width() / 2, m * 100 + 0.3, f"{m:.1%}", ha="center", fontsize=9)
    ax.set_ylabel("No-show rate (%)"); ax.set_title(title); ax.legend(frameon=False)
    plt.xticks(rotation=rot, ha="right" if rot else "center")
    if fname: save(fname)
    return g

print(f"{len(df):,} appointments | {df['patient_id'].nunique():,} patients")

110,521 appointments | 62,298 patients


## 1 · Overall no-show rate

In [2]:
counts = df["missed"].value_counts().rename({0: "Attended", 1: "Missed"})
print(counts.to_string()); print(f"\nNo-show rate: {BASE:.1%}  ->  about 1 in {1/BASE:.0f} appointments is missed")
fig, ax = plt.subplots(figsize=(4.8, 4.8))
ax.pie(counts, labels=counts.index, autopct="%1.1f%%", colors=[BLUE, RED], startangle=90, wedgeprops=dict(width=0.45, edgecolor="white"))
ax.set_title("Attendance outcome"); save("01_overall_noshow")

missed
Attended    88207
Missed      22314

No-show rate: 20.2%  ->  about 1 in 5 appointments is missed


## 2 · Waiting time (days between booking and appointment)

In [3]:
_ = rate_bar("wait_group", "No-show rate rises with waiting time", fname="02_noshow_by_wait_group")
print(df.groupby("wait_group", observed=True).size().to_frame("appointments"))

            appointments
wait_group              
Same day           38562
1-3 days           14675
4-7 days           17510
8-14 days          12025
15-30 days         17371
30+ days           10378


In [4]:
w = df[df["wait_days"] <= 60].groupby("wait_days")["missed"].agg(["mean", "size"])
w = w[w["size"] >= 30]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(w.index, w["mean"] * 100, color=BLUE, marker="o", ms=3)
ax.axhline(BASE * 100, color=RED, ls="--", label="Overall")
ax.set_xlabel("Waiting days"); ax.set_ylabel("No-show rate (%)"); ax.set_title("No-show rate by exact waiting days (<= 60 days)"); ax.legend(frameon=False)
save("02b_noshow_by_wait_days")

In [5]:
same = df[df["wait_days"] == 0]["missed"].mean(); long_ = df[df["wait_days"] > 14]["missed"].mean()
print(f"Same-day appointments : {same:.1%} no-show")
print(f"Waits over 14 days    : {long_:.1%} no-show  ({long_/same:.1f}x higher)")

Same-day appointments : 4.6% no-show
Waits over 14 days    : 32.7% no-show  (7.0x higher)


## 3 · Age and gender

In [6]:
_ = rate_bar("age_group", "No-show rate by age group", rot=30, fname="03_noshow_by_age_group", figsize=(9, 4.5))

In [7]:
a = df[df["age"] <= 95].groupby("age")["missed"].agg(["mean", "size"])
a["smooth"] = a["mean"].rolling(5, center=True, min_periods=1).mean()
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(a.index, a["mean"] * 100, s=12, color=GREY, label="Each age")
ax.plot(a.index, a["smooth"] * 100, color=BLUE, lw=2.5, label="5-year moving average")
ax.axhline(BASE * 100, color=RED, ls="--", label="Overall")
ax.set_xlabel("Age"); ax.set_ylabel("No-show rate (%)"); ax.set_title("Teens and young adults miss most; seniors are the most reliable"); ax.legend(frameon=False)
save("03b_noshow_by_age")

In [8]:
g = rate_bar("gender", "No-show rate by gender", fname="03c_noshow_by_gender", figsize=(4.5, 4))
print(g)

            mean   size
gender                 
F       0.203115  71836
M       0.199638  38685


## 4 · Day of week and booking hour

In [9]:
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
g = rate_bar("appointment_weekday", "No-show rate by appointment weekday", order=days, rot=20, fname="04_noshow_by_weekday")
print(g)

                         mean   size
appointment_weekday                 
Monday               0.206446  22713
Tuesday              0.200874  25638
Wednesday            0.196861  25866
Thursday             0.193494  17246
Friday               0.212261  19019
Saturday             0.230769     39


In [10]:
h = df.groupby("scheduled_hour")["missed"].agg(["mean", "size"]); h = h[h["size"] >= 100]
fig, ax = plt.subplots(figsize=(9, 4)); ax.bar(h.index, h["mean"] * 100, color=BLUE)
ax.axhline(BASE * 100, color=RED, ls="--"); ax.set_xlabel("Hour the appointment was booked"); ax.set_ylabel("No-show rate (%)")
ax.set_title("No-show rate by booking hour"); save("04b_noshow_by_booking_hour")

## 5 · Health conditions and welfare programme

In [11]:
rows = []
for col, label in [("scholarship", "Welfare (Bolsa Familia)"), ("hypertension", "Hypertension"), ("diabetes", "Diabetes"),
                   ("alcoholism", "Alcoholism"), ("handicap", "Handicap"), ("sms_received", "SMS received")]:
    for v in (0, 1):
        s = df[df[col] == v]["missed"]
        rows.append({"factor": label, "has_it": "Yes" if v else "No", "rate": s.mean() * 100, "n": len(s)})
f = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(9, 4.8))
sns.barplot(data=f, y="factor", x="rate", hue="has_it", palette={"No": GREY, "Yes": BLUE}, ax=ax)
ax.axvline(BASE * 100, color=RED, ls="--"); ax.set_xlabel("No-show rate (%)"); ax.set_ylabel("")
ax.set_title("No-show rate: with vs without each factor"); ax.legend(title="", frameon=False)
save("05_noshow_by_flags")
f.pivot(index="factor", columns="has_it", values="rate").round(1)

has_it,No,Yes
factor,,
Alcoholism,20.2,20.1
Diabetes,20.4,18.0
Handicap,20.2,18.1
Hypertension,20.9,17.3
SMS received,16.7,27.6
Welfare (Bolsa Familia),19.8,23.7


In [12]:
g = rate_bar("num_conditions", "No-show rate by number of conditions", fname="05b_noshow_by_conditions", figsize=(6, 4))
print(g)
print("\nNote: patients with chronic conditions are older, and older patients attend more - so condition effects are partly an age effect.")

                    mean   size
num_conditions                 
0               0.209247  84111
1               0.179184  18121
2               0.178637   7658
3               0.153722    618
4               0.307692     13

Note: patients with chronic conditions are older, and older patients attend more - so condition effects are partly an age effect.


In [13]:
# Is the diabetes / hypertension effect still there after controlling for age?
df["age_band"] = pd.cut(df["age"], [-1, 17, 39, 59, 120], labels=["0-17", "18-39", "40-59", "60+"])
t = df.pivot_table(index="age_band", columns="hypertension", values="missed", aggfunc="mean", observed=True) * 100
t.columns = ["No hypertension", "Hypertension"]; t.round(1)

,No hypertension,Hypertension
age_band,,
0-17,21.9,34.2
18-39,23.4,20.1
40-59,19.0,18.3
60+,14.3,16.1


## 6 · SMS reminders - a surprising result
Raw data suggest patients who **received** an SMS miss **more** often. That sounds absurd. Let us check whether it is a confounding effect (*Simpson's paradox*).

In [14]:
raw_sms = df.groupby("sms_received")["missed"].mean() * 100
print("Raw no-show rate  | no SMS: %.1f%%  | SMS: %.1f%%" % (raw_sms[0], raw_sms[1]))
print("\nShare receiving an SMS by wait group:")
print((df.groupby("wait_group", observed=True)["sms_received"].mean() * 100).round(1).to_string())

Raw no-show rate  | no SMS: 16.7%  | SMS: 27.6%

Share receiving an SMS by wait group:
wait_group
Same day       0.0
1-3 days       6.2
4-7 days      60.8
8-14 days     58.2
15-30 days    60.6
30+ days      61.8


In [15]:
g = df.groupby(["wait_group", "sms_received"], observed=True)["missed"].agg(["mean", "size"]).reset_index()
g["rate"] = g["mean"] * 100; g["SMS"] = g["sms_received"].map({0: "No SMS", 1: "SMS sent"})
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=g, x="wait_group", y="rate", hue="SMS", palette={"No SMS": GREY, "SMS sent": BLUE}, ax=ax)
ax.set_xlabel("Waiting time"); ax.set_ylabel("No-show rate (%)"); ax.set_title("SMS effect once waiting time is held constant"); ax.legend(frameon=False)
save("06_sms_effect_by_wait")
g.pivot(index="wait_group", columns="SMS", values="rate").round(1)

SMS,No SMS,SMS sent
wait_group,,
Same day,4.6,NaN
1-3 days,23.0,21.3
4-7 days,27.1,24.0
8-14 days,33.8,28.1
15-30 days,36.9,29.8
30+ days,37.5,30.2


**Interpretation.** SMS reminders are almost never sent for same-day or very short-notice appointments (those patients are already likely to come), and mostly sent for appointments booked far ahead (high-risk). So the SMS group is dominated by long waits, which makes SMS *look* harmful in the raw average.
In **every** waiting-time band where both groups exist, SMS patients miss *less* often (about 2 to 7 percentage points). **The SMS system is not harmful; the raw comparison is confounded** (Simpson's paradox).
Caution: this is observational data. Reminders were probably targeted at certain appointments, so it cannot prove SMS *causes* better attendance. That would need a controlled experiment (A/B test).

## 7 · Neighbourhoods

In [16]:
n = df.groupby("neighbourhood")["missed"].agg(["mean", "size"]).rename(columns={"mean": "rate", "size": "n"})
n = n[n["n"] >= 500].sort_values("rate")
print(f"{len(n)} neighbourhoods with >= 500 appointments (smaller ones are too noisy to rank)")
top = pd.concat([n.head(8), n.tail(8)])
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top.index, top["rate"] * 100, color=[BLUE] * 8 + [RED] * 8)
ax.axvline(BASE * 100, color="black", ls="--", lw=1); ax.set_xlabel("No-show rate (%)")
ax.set_title("Best and worst neighbourhoods (>= 500 appointments)"); save("07_neighbourhood_extremes")
n.tail(5)[::-1]

54 neighbourhoods with >= 500 appointments (smaller ones are too noisy to rank)


,rate,n
neighbourhood,,
SANTOS DUMONT,0.289185,1276
SANTA CLARA,0.264822,506
ITARARÉ,0.262664,3514
JESUS DE NAZARETH,0.243954,2853
ILHA DO PRÍNCIPE,0.234775,2266


In [17]:
# Do the neighbourhood differences exceed random noise? 95% Wilson intervals for the extremes
from statsmodels.stats.proportion import proportion_confint
n["lo"], n["hi"] = proportion_confint((n["rate"] * n["n"]).round(), n["n"], method="wilson")
print(n.tail(3).round(3).to_string()); print(); print(n.head(3).round(3).to_string())

                rate     n     lo     hi
neighbourhood                           
ITARARÉ        0.263  3514  0.248  0.277
SANTA CLARA    0.265   506  0.228  0.305
SANTOS DUMONT  0.289  1276  0.265  0.315

                  rate     n     lo     hi
neighbourhood                             
DO CABRAL        0.157   560  0.129  0.190
SANTA MARTHA     0.158  3131  0.146  0.172
JARDIM DA PENHA  0.163  3877  0.151  0.175


## 8 · Patient history

In [18]:
df["history_group"] = pd.cut(df["prior_noshows"], [-1, 0, 1, 2, 100], labels=["0 missed", "1 missed", "2 missed", "3+ missed"])
g = rate_bar("history_group", "No-show rate by the patient's number of earlier missed appointments", fname="08_noshow_by_history", figsize=(7, 4))
print(g)
print("\nFirst-time vs returning patients:")
print(df.groupby("has_history")["missed"].agg(["mean", "size"]).rename(index={0: "No earlier visit in data", 1: "Has earlier visits"}))

                   mean    size
history_group                  
0 missed       0.200610  100608
1 missed       0.196499    7369
2 missed       0.220049    1636
3+ missed      0.355727     908

First-time vs returning patients:
                              mean   size
has_history                              
No earlier visit in data  0.217888  79642
Has earlier visits        0.160659  30879


The dataset only covers about six weeks of appointments, so most patients have little visible history - a weakness of this dataset. In a real hospital with years of records, patient history is normally one of the strongest predictors.

## 9 · Trend over time

In [19]:
wk = df.groupby("appointment_week").agg(appointments=("missed", "size"), rate=("missed", "mean")).reset_index()
wk = wk[wk["appointments"] > 500]
fig, ax1 = plt.subplots(figsize=(9, 4)); ax1.bar(wk["appointment_week"], wk["appointments"], width=5, color="#D6E6F0")
ax1.set_ylabel("Appointments"); ax2 = ax1.twinx(); ax2.plot(wk["appointment_week"], wk["rate"] * 100, color=RED, marker="o")
ax2.set_ylabel("No-show rate (%)"); ax2.set_ylim(0, 30); ax2.grid(False); ax1.set_title("Weekly appointment volume and no-show rate")
plt.setp(ax1.get_xticklabels(), rotation=25); save("09_weekly_trend")

## 10 · Correlations

In [20]:
cols = ["missed", "age", "wait_days", "scholarship", "hypertension", "diabetes", "alcoholism", "handicap", "sms_received",
        "num_conditions", "prior_appts", "prior_noshows", "scheduled_hour", "is_female"]
corr = df[cols].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax, annot_kws={"size": 8}, cbar_kws={"shrink": .8})
ax.set_title("Correlation matrix"); save("10_correlation_heatmap")
corr["missed"].drop("missed").sort_values(key=abs, ascending=False).round(3).to_frame("corr with missed")

,corr with missed
wait_days,0.186
sms_received,0.127
scheduled_hour,0.061
age,-0.060
prior_appts,-0.046
hypertension,-0.036
prior_noshows,0.035
num_conditions,-0.031
scholarship,0.029
diabetes,-0.015


No single variable has a strong linear relationship with missing (all |r| < 0.2). Waiting time and age matter most. This means prediction will be useful but never perfect - behaviour is only partly explained by the recorded data.

## 11 · Statistical tests
* **Chi-square test of independence** for yes/no and categorical factors, with **Cramer's V** as effect size
* **Mann-Whitney U** for numeric variables (age, wait days) because they are skewed

With 110k rows almost everything is "significant", so the **effect size** is what tells us whether a factor matters in practice.

In [21]:
def chi_test(col):
    ct = pd.crosstab(df[col], df["missed"]); chi2, p, dof, _ = stats.chi2_contingency(ct)
    v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
    return {"factor": col, "test": "Chi-square", "statistic": chi2, "p_value": p, "effect_size": v}

res = [chi_test(c) for c in ["gender", "scholarship", "hypertension", "diabetes", "alcoholism", "handicap",
                             "sms_received", "appointment_weekday", "age_group", "wait_group", "neighbourhood", "has_history"]]
for c in ["age", "wait_days", "scheduled_hour"]:
    a, b = df.loc[df["missed"] == 1, c], df.loc[df["missed"] == 0, c]
    u, p = stats.mannwhitneyu(a, b); r = 1 - 2 * u / (len(a) * len(b))   # rank-biserial correlation
    res.append({"factor": c, "test": "Mann-Whitney U", "statistic": u, "p_value": p, "effect_size": abs(r)})

tests = pd.DataFrame(res).sort_values("effect_size", ascending=False)
tests["significant (p<0.05)"] = np.where(tests["p_value"] < 0.05, "Yes", "No")
tests["strength"] = pd.cut(tests["effect_size"], [-1, 0.05, 0.1, 0.3, 1], labels=["negligible", "small", "medium", "large"])
tests.to_csv("../reports/statistical_tests.csv", index=False)
tests.assign(p_value=tests["p_value"].map(lambda x: f"{x:.2e}"), statistic=tests["statistic"].round(1), effect_size=tests["effect_size"].round(3))

,factor,test,statistic,p_value,effect_size,significant (p<0.05),strength
13,wait_days,Mann-Whitney U,1.374366e+09,0.00e+00,0.397,Yes,large
9,wait_group,Chi-square,9.624000e+03,0.00e+00,0.295,Yes,medium
6,sms_received,Chi-square,1.768000e+03,0.00e+00,0.126,Yes,medium
14,scheduled_hour,Mann-Whitney U,1.075651e+09,8.54e-104,0.093,Yes,small
8,age_group,Chi-square,8.597000e+02,2.47e-181,0.088,Yes,small
12,age,Mann-Whitney U,8.974724e+08,4.27e-92,0.088,Yes,small
10,neighbourhood,Chi-square,4.930000e+02,9.76e-61,0.067,Yes,small
11,has_history,Chi-square,4.519000e+02,2.77e-100,0.064,Yes,small
2,hypertension,Chi-square,1.403000e+02,2.25e-32,0.036,Yes,negligible
1,scholarship,Chi-square,9.380000e+01,3.54e-22,0.029,Yes,negligible


In [22]:
# Effect size summary chart
t = tests.sort_values("effect_size")
fig, ax = plt.subplots(figsize=(8, 5)); ax.barh(t["factor"], t["effect_size"], color=BLUE)
ax.set_xlabel("Effect size (Cramer's V / rank-biserial r)"); ax.set_title("How strongly is each factor linked to missing an appointment?")
save("11_effect_sizes")

In [23]:
# Odds ratios: logistic regression with all factors together.
# Waiting time and age are entered as BANDS (not straight lines) so the model can capture their non-linear effect.
import statsmodels.api as sm
X = pd.concat([
    df[["is_female", "scholarship", "hypertension", "diabetes", "alcoholism", "handicap", "sms_received"]],
    df[["prior_noshows"]].clip(upper=5),
    pd.get_dummies(df["wait_group"], prefix="wait", drop_first=True).astype(float),
    pd.get_dummies(df["age_group"], prefix="age", drop_first=True).astype(float)], axis=1).astype(float)
m = sm.Logit(df["missed"], sm.add_constant(X)).fit(disp=0)
ci = np.exp(m.conf_int())
orr = pd.DataFrame({"odds_ratio": np.exp(m.params), "ci_low": ci[0], "ci_high": ci[1], "p_value": m.pvalues}).drop("const")
print("Reference groups: same-day appointment, infant (0-2). Odds ratio > 1 = higher risk of missing, < 1 = lower risk.")
orr.round(3)

Reference groups: same-day appointment, infant (0-2). Odds ratio > 1 = higher risk of missing, < 1 = lower risk.


,odds_ratio,ci_low,ci_high,p_value
is_female,0.950,0.918,0.983,0.003
scholarship,1.233,1.171,1.298,0.000
hypertension,0.992,0.943,1.044,0.762
diabetes,1.158,1.081,1.242,0.000
alcoholism,1.363,1.240,1.497,0.000
handicap,1.079,0.961,1.212,0.199
sms_received,0.758,0.732,0.786,0.000
prior_noshows,1.421,1.378,1.465,0.000
wait_1-3 days,7.046,6.620,7.498,0.000
wait_4-7 days,8.959,8.412,9.541,0.000


**Reading the odds ratios (all factors adjusted for each other)**
- **Waiting time** dominates: compared with a same-day appointment, waits of 15+ days have roughly 13-14x the odds of being missed.
- **SMS reminder: odds ratio about 0.76** - once waiting time is accounted for, an SMS is linked to about 24% lower odds of missing. This confirms the Simpson's-paradox explanation above.
- **Age:** teens and young adults are the riskiest; seniors (60+) have much lower odds than any other group.
- **Hypertension has no independent effect** (CI includes 1). Its "better attendance" in the raw numbers was an age effect.
- **Diabetes (odds ratio about 1.16) and alcoholism (about 1.36) are *higher* risk** once age is controlled. In the raw numbers they looked neutral or protective only because these patients are older. Alcoholism is a small group, so its estimate is less precise.
- **Prior no-shows** raise the odds (about 1.4x per earlier missed visit), and **welfare enrolment** raises them by about 23%.
- Treat these as *associations*, not causes.

## 12 · Key findings
1. **About 1 in 5 appointments (20.2%) is missed.**
2. **Waiting time is the dominant factor.** Same-day appointments are missed far less than appointments booked more than two weeks ahead.
3. **Teenagers and young adults (about 13-34) miss most (roughly 23-27%); seniors 60+ are the most reliable (about 15%).**
4. **Gender and weekday matter very little.**
5. **Welfare (scholarship) enrolment is linked to higher risk (23.7% vs 19.8%).** Hypertension looks protective in raw data, but this is purely an age effect; after adjusting for age, diabetes and alcoholism are linked to *higher* risk.
6. **SMS looks harmful in the raw data (27.6% vs 16.7%) only because reminders go mostly to long-wait appointments.** Within each waiting band, SMS patients miss less (odds ratio about 0.76).
7. **Some neighbourhoods have clearly higher no-show rates** (for example Santos Dumont, Itarare), but small ones are too noisy to rank.
8. **Effects are individually modest** (all effect sizes small), so we combine factors in a predictive model in notebook 03. Prior no-show history is weaker here than it would be in a real hospital, because the data covers only about six weeks.